# Data Engineering — FVC2004

Builds the dataset manifest, splits by finger identity (no data leakage), and generates genuine/impostor verification pairs for train, val, and test sets.

## Dataset layout (local)

- `dataset/FVC2004/DB1_A` … `DB4_B` — eight subsets (4 sensors × {A, B})
- **A-set**: 100 fingers × 8 impressions (800 images per DB)
- **B-set**: 10 fingers × 8 impressions (80 images per DB); finger indices start at **101**

Filenames: `{finger_index}_{impression}.tif` (e.g. `5_3.tif`).

## Finger identity (verification label)

A **finger identity** is one physical finger in one FVC subset:

- `finger_id = "{DBx_subset}_{finger_index}"` — e.g. `DB2_A_042`, `DB1_B_105`

Fingers in different subsets (different sensors or A vs B) are **different identities**, even if numeric indices match.

## Outputs

Written under **`artifacts/`**:

- `manifest.csv`, `manifest_with_split.csv`
- `pairs_train.csv`, `pairs_val.csv`, `pairs_test.csv`
- `dataset_summary.json`

In [1]:
# Imports and project path setup
from __future__ import annotations

import csv
import json
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from typing import Iterator

import numpy as np
from PIL import Image

_PROJECT_CWD = Path.cwd()
if _PROJECT_CWD.name == "notebooks":
    PROJECT_ROOT = _PROJECT_CWD.parent
else:
    PROJECT_ROOT = _PROJECT_CWD

DATA_ROOT = PROJECT_ROOT / "dataset" / "FVC2004"
OUT_DIR = PROJECT_ROOT / "artifacts"

SEED = 42
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Pair sampling (same spirit as SOCOFing notebook)
IMPOSTOR_PER_GENUINE = 1.0
MAX_GENUINE_PAIRS_PER_FINGER = 28  # 8 impressions → at most C(8,2)=28 genuine pairs

random.seed(SEED)
np.random.seed(SEED)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR:", OUT_DIR)

PROJECT_ROOT: /Users/spartan/Desktop/CS228/Project
DATA_ROOT: /Users/spartan/Desktop/CS228/Project/dataset/FVC2004
OUT_DIR: /Users/spartan/Desktop/CS228/Project/artifacts


In [2]:
# FVC2004 partition names and sensor metadata

FVC_PARTITIONS: tuple[str, ...] = tuple(
    f"DB{i}_{letter}" for i in range(1, 5) for letter in ("A", "B")
)

SENSOR_BY_DB: dict[int, str] = {
    1: "optical_crossmatch_v300",
    2: "optical_digitalpersona_uareu4000",
    3: "thermal_atmel_fingerchip",
    4: "synthetic_sfinge",
}

for name in FVC_PARTITIONS:
    p = DATA_ROOT / name
    assert p.is_dir(), f"Missing folder: {p}"
    n = len(list(p.glob("*.tif")))
    db_num = int(name.split("_")[0][2:])
    print(f"{name}: {n} images  sensor={SENSOR_BY_DB[db_num]}")

DB1_A: 800 images  sensor=optical_crossmatch_v300
DB1_B: 80 images  sensor=optical_crossmatch_v300
DB2_A: 800 images  sensor=optical_digitalpersona_uareu4000
DB2_B: 80 images  sensor=optical_digitalpersona_uareu4000
DB3_A: 800 images  sensor=thermal_atmel_fingerchip
DB3_B: 80 images  sensor=thermal_atmel_fingerchip
DB4_A: 800 images  sensor=synthetic_sfinge
DB4_B: 80 images  sensor=synthetic_sfinge


In [3]:
# Parse filenames and define the image dataclass
FNAME_RE = re.compile(r"^(?P<finger>\d+)_(?P<imp>\d+)\.tif$", re.IGNORECASE)


@dataclass(frozen=True)
class ParsedImage:
    partition: str  # e.g. DB1_A
    db_number: int
    subset: str  # A or B
    finger_index: int
    impression: int
    sensor: str

    @property
    def finger_id(self) -> str:
        return f"{self.partition}_{self.finger_index:03d}"


def parse_tif_name(fname: str) -> tuple[int, int]:
    m = FNAME_RE.match(fname)
    if not m:
        raise ValueError(f"Unrecognized FVC filename: {fname}")
    return int(m.group("finger")), int(m.group("imp"))


def iter_partition_images(partition: str) -> Iterator[ParsedImage]:
    folder = DATA_ROOT / partition
    db_number = int(partition.split("_")[0][2:])
    subset = partition.split("_")[1]
    sensor = SENSOR_BY_DB[db_number]
    for path in sorted(folder.glob("*.tif")):
        finger, imp = parse_tif_name(path.name)
        yield ParsedImage(partition, db_number, subset, finger, imp, sensor)


# Sanity on a few names
for ex in ["101_1.tif", "5_8.tif", "100_1.tif"]:
    print(ex, parse_tif_name(ex))

101_1.tif (101, 1)
5_8.tif (5, 8)
100_1.tif (100, 1)


In [5]:
# Scan the dataset directory and build the image manifest
def relpath(p: Path) -> str:
    return str(p.relative_to(PROJECT_ROOT).as_posix())


rows: list[dict] = []

for part in FVC_PARTITIONS:
    for info in iter_partition_images(part):
        p = DATA_ROOT / part / f"{info.finger_index}_{info.impression}.tif"
        rows.append(
            {
                "image_path": relpath(p),
                "partition": info.partition,
                "db_number": info.db_number,
                "subset": info.subset,
                "sensor": info.sensor,
                "finger_index": info.finger_index,
                "impression": info.impression,
                "finger_id": info.finger_id,
            }
        )

print("Total images:", len(rows))
print("Unique Fingers:", len({r["finger_id"] for r in rows}))
print("By partition:", dict(Counter(r["partition"] for r in rows)))

Total images: 3520
Unique Fingers: 440
By partition: {'DB1_A': 800, 'DB1_B': 80, 'DB2_A': 800, 'DB2_B': 80, 'DB3_A': 800, 'DB3_B': 80, 'DB4_A': 800, 'DB4_B': 80}


In [6]:
# Verify each finger has exactly 8 impressions
by_finger: dict[str, list[dict]] = defaultdict(list)
for r in rows:
    by_finger[r["finger_id"]].append(r)

imp_counts = sorted(len(v) for v in by_finger.values())
print("Finger identities:", len(by_finger))
print("Impressions per finger: min / mean / max =", min(imp_counts), sum(imp_counts) / len(imp_counts), max(imp_counts))
if len(set(imp_counts)) == 1:
    print("All fingers have the same impression count:", imp_counts[0])
else:
    print("Impression count histogram:", dict(Counter(imp_counts)))

# Native image sizes by partition (one sample per partition)
print("\nSample (W,H) by partition:")
for part in FVC_PARTITIONS:
    sample = DATA_ROOT / part / sorted((DATA_ROOT / part).glob("*.tif"))[0]
    with Image.open(sample) as img:
        print(f"  {part}: {img.size}, mode={img.mode}")

Finger identities: 440
Impressions per finger: min / mean / max = 8 8.0 8
All fingers have the same impression count: 8

Sample (W,H) by partition:
  DB1_A: (640, 480), mode=L
  DB1_B: (640, 480), mode=L
  DB2_A: (328, 364), mode=L
  DB2_B: (328, 364), mode=L
  DB3_A: (300, 480), mode=L
  DB3_B: (300, 480), mode=L
  DB4_A: (288, 384), mode=L
  DB4_B: (288, 384), mode=L


In [7]:
# Split fingers into train/val/test — all impressions of a finger stay in one split


def allocate_counts(n: int, train_ratio: float, val_ratio: float, test_ratio: float) -> tuple[int, int, int]:
    s = train_ratio + val_ratio + test_ratio
    if abs(s - 1.0) > 1e-9:
        raise ValueError("Ratios must sum to 1")
    n_train = int(round(n * train_ratio))
    n_val = int(round(n * val_ratio))
    n_test = n - n_train - n_val
    if n >= 10:
        if n_val == 0:
            n_val, n_train = n_val + 1, n_train - 1
        if n_test == 0:
            n_test, n_train = n_test + 1, n_train - 1
    while n_train + n_val + n_test < n:
        n_train += 1
    return n_train, n_val, n_test


finger_ids = sorted(by_finger.keys())
rng = random.Random(SEED)
rng.shuffle(finger_ids)

n_train, n_val, n_test = allocate_counts(len(finger_ids), TRAIN_RATIO, VAL_RATIO, TEST_RATIO)
train_f = set(finger_ids[:n_train])
val_f = set(finger_ids[n_train : n_train + n_val])
test_f = set(finger_ids[n_train + n_val : n_train + n_val + n_test])

print("Split finger_id counts — train:", len(train_f), " val:", len(val_f), " test:", len(test_f))

rows_with_split: list[dict] = []
for r in rows:
    fid = r["finger_id"]
    if fid in train_f:
        s = "train"
    elif fid in val_f:
        s = "val"
    else:
        s = "test"
    rr = dict(r)
    rr["split"] = s
    rows_with_split.append(rr)

print("Images by split:", dict(Counter(r["split"] for r in rows_with_split)))

Split finger_id counts — train: 308  val: 66  test: 66
Images by split: {'train': 2464, 'test': 528, 'val': 528}


In [8]:
# Preprocessing function — convert to grayscale, resize, normalize to [0, 1]


def load_and_preprocess(path: Path, size: tuple[int, int] = (256, 256)) -> np.ndarray:
    with Image.open(path) as img:
        arr = np.asarray(img.convert("L").resize(size), dtype=np.float32) / 255.0
    return arr


_demo_paths = [PROJECT_ROOT / rows_with_split[i]["image_path"] for i in rng.sample(range(len(rows_with_split)), min(5, len(rows_with_split)))]
for p in _demo_paths:
    x = load_and_preprocess(p)
    print(p.name, x.shape, f"min={x.min():.3f} max={x.max():.3f} mean={x.mean():.3f}")

1_2.tif (256, 256) min=0.325 max=1.000 mean=0.790
85_2.tif (256, 256) min=0.012 max=1.000 mean=0.891
81_5.tif (256, 256) min=0.000 max=0.902 mean=0.598
87_1.tif (256, 256) min=0.000 max=0.839 mean=0.532
91_6.tif (256, 256) min=0.000 max=1.000 mean=0.964


In [9]:
# Generate genuine and impostor pairs within each split

by_split_finger: dict[str, dict[str, list[str]]] = {
    "train": defaultdict(list),
    "val": defaultdict(list),
    "test": defaultdict(list),
}
for r in rows_with_split:
    by_split_finger[r["split"]][r["finger_id"]].append(r["image_path"])


def sample_genuine_pairs(image_paths: list[str], cap: int) -> list[tuple[str, str]]:
    imgs = sorted(image_paths)
    all_pairs = list(combinations(imgs, 2))
    if len(all_pairs) <= cap:
        return all_pairs
    return rng.sample(all_pairs, cap)


def build_pairs_for_split(split: str) -> tuple[list[dict], dict]:
    fingers = by_split_finger[split]

    genuine: list[tuple[str, str, str]] = []
    for fid, imgs in fingers.items():
        for a, b in sample_genuine_pairs(imgs, cap=MAX_GENUINE_PAIRS_PER_FINGER):
            genuine.append((a, b, fid))

    impostor_target = int(round(len(genuine) * IMPOSTOR_PER_GENUINE))
    finger_list = sorted(fingers.keys())

    impostor: list[tuple[str, str, str, str]] = []
    seen: set[tuple[str, str]] = set()
    max_attempts = max(50_000, impostor_target * 30)
    attempts = 0
    while len(impostor) < impostor_target and attempts < max_attempts:
        attempts += 1
        f1, f2 = rng.sample(finger_list, 2)
        a, b = rng.choice(fingers[f1]), rng.choice(fingers[f2])
        key = tuple(sorted((a, b)))
        if key in seen:
            continue
        seen.add(key)
        impostor.append((a, b, f1, f2))

    pair_rows: list[dict] = []
    for a, b, fid in genuine:
        pair_rows.append(
            {"image_path_1": a, "image_path_2": b, "label": 1, "finger_id_1": fid, "finger_id_2": fid}
        )
    for a, b, f1, f2 in impostor:
        pair_rows.append(
            {"image_path_1": a, "image_path_2": b, "label": 0, "finger_id_1": f1, "finger_id_2": f2}
        )
    rng.shuffle(pair_rows)

    summary = {
        "split": split,
        "genuine_pairs": len(genuine),
        "impostor_pairs": len(impostor),
        "total_pairs": len(pair_rows),
        "impostor_sampling_attempts": attempts,
    }
    return pair_rows, summary


pairs_train, sum_train = build_pairs_for_split("train")
pairs_val, sum_val = build_pairs_for_split("val")
pairs_test, sum_test = build_pairs_for_split("test")
sum_train, sum_val, sum_test

({'split': 'train',
  'genuine_pairs': 8624,
  'impostor_pairs': 8624,
  'total_pairs': 17248,
  'impostor_sampling_attempts': 8634},
 {'split': 'val',
  'genuine_pairs': 1848,
  'impostor_pairs': 1848,
  'total_pairs': 3696,
  'impostor_sampling_attempts': 1869},
 {'split': 'test',
  'genuine_pairs': 1848,
  'impostor_pairs': 1848,
  'total_pairs': 3696,
  'impostor_sampling_attempts': 1864})

In [ ]:
# Save manifest, split assignments, pair CSVs, and dataset summary to artifacts/

OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest_fields = [
    "image_path",
    "partition",
    "db_number",
    "subset",
    "sensor",
    "finger_index",
    "impression",
    "finger_id",
]
manifest_split_fields = manifest_fields + ["split"]
pair_fields = ["image_path_1", "image_path_2", "label", "finger_id_1", "finger_id_2"]


def write_csv(path: Path, fieldnames: list[str], data: list[dict]) -> None:
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in data:
            w.writerow({k: r.get(k, "") for k in fieldnames})


write_csv(OUT_DIR / "manifest.csv", manifest_fields, rows)
write_csv(OUT_DIR / "manifest_with_split.csv", manifest_split_fields, rows_with_split)
write_csv(OUT_DIR / "pairs_train.csv", pair_fields, pairs_train)
write_csv(OUT_DIR / "pairs_val.csv", pair_fields, pairs_val)
write_csv(OUT_DIR / "pairs_test.csv", pair_fields, pairs_test)

img_by_split = Counter(r["split"] for r in rows_with_split)
imp_per = [len(by_finger[fid]) for fid in by_finger]

summary = {
    "dataset": "FVC2004",
    "dataset_root": str(DATA_ROOT.relative_to(PROJECT_ROOT).as_posix()),
    "num_images_total": len(rows),
    "num_finger_identities": len(by_finger),
    "partitions": list(FVC_PARTITIONS),
    "images_by_partition": dict(Counter(r["partition"] for r in rows)),
    "images_by_sensor": dict(Counter(r["sensor"] for r in rows)),
    "impressions_per_finger_min": int(min(imp_per)),
    "impressions_per_finger_max": int(max(imp_per)),
    "impressions_per_finger_mean": float(sum(imp_per) / len(imp_per)),
    "split": {
        "finger_id_counts": {"train": len(train_f), "val": len(val_f), "test": len(test_f)},
        "image_counts": dict(img_by_split),
    },
    "pairs": {"train": sum_train, "val": sum_val, "test": sum_test},
    "preprocessing": {"convert": "grayscale", "resize": [256, 256], "normalize": "x/255"},
    "config": {
        "seed": SEED,
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO,
        "impostor_per_genuine": IMPOSTOR_PER_GENUINE,
        "max_genuine_pairs_per_finger": MAX_GENUINE_PAIRS_PER_FINGER,
    },
}

(OUT_DIR / "dataset_summary.json").write_text(json.dumps(summary, indent=2))

print("Wrote:")
for p in [
    OUT_DIR / "manifest.csv",
    OUT_DIR / "manifest_with_split.csv",
    OUT_DIR / "pairs_train.csv",
    OUT_DIR / "pairs_val.csv",
    OUT_DIR / "pairs_test.csv",
    OUT_DIR / "dataset_summary.json",
]:
    print(" -", p)